# Project — Airline AI Assistant (trợ lý AI hàng không)

Bây giờ chúng ta ghép những gì đã học để làm AI Customer Support (hỗ trợ khách hàng) cho một hãng hàng không.

In [ ]:
# imports (nhập thư viện)

import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import sqlite3

In [ ]:
# Khởi tạo

load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key tồn tại và bắt đầu bằng {openai_api_key[:8]}")
else:
    print("OpenAI API Key chưa được đặt")
    
MODEL = "gpt-4.1-mini"
openai = OpenAI()

DB = "prices.db"

In [ ]:
system_message = """
Bạn là trợ lý hữu ích của hãng hàng không FlightAI.
Trả lời ngắn, lịch sự, không quá 1 câu.
Luôn chính xác. Nếu không biết đáp án, hãy nói vậy.
"""

In [ ]:
# Tra giá vé từ SQLite (database file, không cần server riêng)

def get_ticket_price(city):
    print(f"DATABASE TOOL ĐƯỢC GỌI: Lấy giá vé cho {city}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT price FROM prices WHERE city = ?', (city.lower(),))
        result = cursor.fetchone()
        return f"Giá vé đến {city} là ${result[0]}" if result else "Không có dữ liệu giá cho thành phố này"

In [ ]:
# Thử gọi tool trực tiếp (chưa qua LLM)

get_ticket_price("Paris")

In [ ]:
# Dictionary mô tả function để LLM biết khi nào nên gọi tool

price_function = {
    "name": "get_ticket_price",
    "description": "Lấy giá vé khứ hồi đến thành phố đích. Truyền tên thành phố bằng tiếng Anh (ví dụ: London, Paris, Tokyo, Berlin).",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "Thành phố khách muốn đến, viết bằng tiếng Anh",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}
tools = [{"type": "function", "function": price_function}]
tools

In [ ]:
# Chatbot Gradio chưa có tools — chỉ trả lời từ system_message

def chat(message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content

gr.ChatInterface(fn=chat, type="messages").launch()

In [ ]:
# Vòng while: model có thể gọi tool nhiều lượt liên tiếp trước khi trả lời user

def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    
    return response.choices[0].message.content

In [ ]:
# Xử lý mọi tool_call trong cùng một message (không chỉ cái đầu)

def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            price_details = get_ticket_price(city)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
    return responses

In [ ]:
# Chatbot Gradio có tools — tra giá vé từ database

gr.ChatInterface(fn=chat, type="messages").launch()

## Gradio thực sự làm gì:

1. Gradio dựng frontend Svelte app dựa trên mô tả UI bằng Python của chúng ta
2. Gradio khởi động server trên Starlette web framework, lắng nghe một cổng (port) trống và phục vụ Svelte app đó
3. Gradio tạo backend routes cho các callback, ví dụ `chat()`, rồi gọi function của chúng ta

Và khi Gradio sinh frontend app, nó đảm bảo nút Submit gọi đúng backend route.

Vậy thôi!

Đơn giản, nhưng kết quả cảm giác như phép thuật.

# Chuyển sang multi-modal (đa phương thức)!!

Chúng ta dùng `gpt-image-1-mini`, model tạo ảnh đứng sau GPT-5, để vẽ một số hình.

Đặt logic này vào function tên `artist`.

### Cảnh báo giá: mỗi lần tạo ảnh tốn khoảng 3 cents — đừng tạo ảnh quá nhiều!

In [ ]:
# Một số import để xử lý ảnh

import base64
from io import BytesIO
from PIL import Image

In [ ]:
# Tạo ảnh minh họa kỳ nghỉ tại thành phố (prompt tiếng Anh để model ảnh hiểu tốt hơn)

def artist(city):
    image_response = openai.images.generate(
            model="gpt-image-1-mini",
            prompt=f"An image representing a vacation in {city}, showing tourist spots and everything unique about {city}, in a vibrant pop-art style",
            size="1024x1024",
            n=1,
        )
    image_base64 = image_response.data[0].b64_json
    image_data = base64.b64decode(image_base64)
    return Image.open(BytesIO(image_data))

In [ ]:
# Thử tạo ảnh New York City

image = artist("New York City")
display(image)

In [ ]:
# TTS (text-to-speech): chuyển câu trả lời thành âm thanh

def talker(message):
    response = openai.audio.speech.create(
      model="gpt-4o-mini-tts",
      voice="onyx",    # Thử thay onyx bằng alloy hoặc coral
      input=message
    )
    return response.content

## Ghép tất cả lại:

1. Trợ lý AI multi-modal với tạo ảnh và audio
2. Tool calling kèm tra cứu database
3. Một bước tiến tới Agentic workflow (luồng có tính agent)


In [ ]:
# chat() điều phối: tool → câu trả lời → TTS → ảnh; trả về history, voice, image

def chat(history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    cities = []
    image = None

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses, cities = handle_tool_calls_and_return_cities(message)
        messages.append(message)
        messages.extend(responses)
        response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    reply = response.choices[0].message.content
    history += [{"role":"assistant", "content":reply}]

    voice = talker(reply)

    if cities:
        image = artist(cities[0])
    
    return history, voice, image


In [ ]:
# Chạy tool và đồng thời trả về danh sách thành phố (để artist vẽ ảnh)

def handle_tool_calls_and_return_cities(message):
    responses = []
    cities = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            cities.append(city)
            price_details = get_ticket_price(city)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
    return responses, cities

## 3 kiểu UI của Gradio

`gr.Interface` dùng cho UI chuẩn, đơn giản

`gr.ChatInterface` dùng cho UI ChatBot chuẩn

`gr.Blocks` dùng cho UI tùy chỉnh: tự kiểm soát component và callback

In [ ]:
# Callbacks (cùng với function chat() ở trên)

def put_message_in_chatbot(message, history):
        return "", history + [{"role":"user", "content":message}]

# Định nghĩa UI

with gr.Blocks() as ui:
    with gr.Row():
        chatbot = gr.Chatbot(height=500, type="messages")
        image_output = gr.Image(height=500, interactive=False)
    with gr.Row():
        audio_output = gr.Audio(autoplay=True)
    with gr.Row():
        message = gr.Textbox(label="Chat với trợ lý AI của chúng ta:")

# Nối sự kiện với callbacks

    message.submit(put_message_in_chatbot, inputs=[message, chatbot], outputs=[message, chatbot]).then(
        chat, inputs=chatbot, outputs=[chatbot, audio_output, image_output]
    )

ui.launch(inbrowser=True, auth=("ed", "bananas"))

# Bài tập và ứng dụng thực tế

Thêm nhiều tools — ví dụ mô phỏng đặt vé máy bay. Một học viên đã làm và chia sẻ ví dụ trong thư mục community contributions.

Tiếp theo: áp dụng vào công việc của bạn. Làm trợ lý AI multi-modal với tools thực hiện một việc trong công việc. Trợ lý hỗ trợ khách hàng? Trợ lý onboarding nhân viên mới? Rất nhiều khả năng! Cũng xem bài tập cuối week2 trong notebook riêng.

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Mình có một lời nhờ đặc biệt</h2>
            <span style="color:#090;">
                Biên tập viên của mình nói rằng việc học viên đánh giá khóa học trên Udemy tạo ra sự khác biệt RẤT lớn — đó là một trong những cách chính để Udemy quyết định có giới thiệu khóa học cho người khác hay không. Nếu bạn dành một phút để đánh giá, mình sẽ rất biết ơn! Và dù sao — luôn vui lòng liên hệ ed@edwarddonner.com nếu mình có thể giúp bất kỳ lúc nào.
            </span>
        </td>
    </tr>
</table>